### 手術室準備檢查

在急救或多台手術交替進行的情境中，醫護人員可能要快速判斷每個手術室的情況。刀具的擺設可以幫助他們快速了解進行中的手術種類，以便分配適當的支援或後續工作。

**術前確認會議**：在一些大型手術中，手術團隊會進行術前會議（稱為 time-out 或術前暫停），醫師會確認刀具和設備的擺設，檢查與該手術需求是否吻合。例如，心臟手術需要精細的縫合工具和血管夾，若發現缺少這些工具，醫護人員會及時補充，避免手術中斷。

**手術類型的變更**：有時病人狀況可能在手術中改變，必須更改手術計劃，導致需要快速增加或更換手術工具。此時，可以根據刀具的擺放情況，判斷是否有符合新手術需求的器械，或是通知其他醫護人員增援。

**設備檢查員的交接**：在手術前，設備檢查員可能需要進行交接，下一班次的檢查員透過檢查刀具和設備的擺設，可以快速了解當前手術需求及預計進行的步驟。

In [1]:
import re

cls_labels ={0: 'Knife_Handle',
 1: 'Metzenbaum',
 2: 'Needle',
 3: 'Needle_Holder',
 4: 'Suture_Scissor',
 5: 'Smooth_Forceps',
 6: 'Teeth_Forceps',
 7: 'mosquito',
 8: 'thread'}

code_to_names: {'A':'Knife_Handle',
 'B': 'Metzenbaum',
 'C': 'Needle',
 'D': 'Needle_Holder',
 'E': 'Suture_Scissor',
 'F': 'Smooth_Forceps',
 'G': 'Teeth_Forceps',
 'H': 'mosquito',
 'I': 'thread'}

names_to_code = {'Knife_Handle':'A',
 'Metzenbaum':'B',
 'Needle':'C',
 'Needle_Holder':'D',
 'Suture_Scissor':'E',
 'Smooth_Forceps':'F',
 'Teeth_Forceps':'G',
 'mosquito':'H',
 'thread':'I'
}

# code_to_surgery_name = {'AHBFDE':'Skin Surgery',
#  'AHFBDE':'Open Surgery'
# }


def code_to_surgery_name(code):
    # 定義模式
    patterns = {
        'Surgery_1': r'^AHGBDE$',                         # 完全匹配 AHGBDE
        'Surgery_2': r'^AHBGE$',                          # 完全匹配 AHBGE
        'Surgery_3': r'^AH{2,4}BG{1,2}DE$',               # H 出現 2 到 4 次, G 出現 1 到 2 次
        'Surgery_4': r'^AH{4,6}G{2,4}BDE$'                # H 出現 4 到 6 次, G 出現 2 到 4 次
    }
    
    # 逐個模式進行匹配
    for surgery, pattern in patterns.items():
        if re.match(pattern, code):
            return surgery

    return "UNKNOWN"

# # 測試
# test_codes = ['AHGBDE', 'AHBGE', 'AHHHHBGGBDE', 'AHHHHHHGGGGBDE', 'AHHHBGGBDE']
# for code in test_codes:
#     print(f"{code}: {classify_surgery(code)}")

In [2]:
model_path = '/.../.../train3/weights/best.pt'
output_video_name = 'real_time_output.mp4'

In [3]:
def generate_code(data, names_to_code):
    code = ''
    for item in data:
        # item[4] 是工具名称
        name = item[4]
        if name in names_to_code:
            code += names_to_code[name]
    return code


## Session 2
「我的result裡面有一些instance會有xy座標太相近以致會有重疊的情況，

  我希望能以conf為依據，來篩掉conf比較小的那個結果」

In [4]:
# 非極大值抑制函數
def non_max_suppression(boxes, conf_threshold=0.5, iou_threshold=0.4):
    if len(boxes) == 0:
        return []

    # 先將 boxes 按照置信度排序
    boxes = sorted(boxes, key=lambda x: x['confidence'], reverse=True)
    selected_boxes = []

    while boxes:
        # 取出置信度最高的框
        current_box = boxes.pop(0)
        selected_boxes.append(current_box)

        # 計算與其他框的 IoU 並篩選
        boxes = [
            box for box in boxes
            if iou(current_box['bbox'], box['bbox']) < iou_threshold
        ]

    return selected_boxes

# IoU 計算函數
def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    
    # 計算交集區域的面積
    interArea = max(0, xB - xA) * max(0, yB - yA)
    if interArea == 0:
        return 0.0

    # 計算各自框的面積
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    # 計算 IoU
    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou

In [5]:
# 僅使用非極大值抑制來篩選框

import cv2
from ultralytics import YOLO
import time
import copy

cap = cv2.VideoCapture(4)
model = YOLO(model_path)
order_status = "UNKNOWN"

# 設置錄製的編碼格式和參數（這裡使用 XVID 編碼器）
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = None  # 初始化錄製變量
recording = False  # 是否正在錄製的標誌

print("按下 'a' 開始錄製，'q' 停止錄製並退出")

dete_hist = []


while cap.isOpened():
    ret, org_frame = cap.read()
    if not ret: break
    
    frame = copy.deepcopy(org_frame)
    height, width = frame.shape[:2]

    results = model(frame, iou=0.7)

    # 取得偵測框並將其整理成適合 NMS 的格式
    boxes = []
        
    # 清空舊的偵測結果
    detections = []

    for result in results:
        for box in result.boxes:
            # 取得邊框座標與標籤
            x1, y1, x2, y2 = map(int, box.xyxy[0])  # 取得框座標
            label = result.names[int(box.cls[0])]
            confidence = box.conf[0]
            boxes.append({
                'bbox': (x1, y1, x2, y2),
                'confidence': confidence,
                'label': label
            })

    # 使用非極大值抑制來篩選框
    filtered_boxes = non_max_suppression(boxes)
    
            
    for box in filtered_boxes:
        x1, y1, x2, y2 = box['bbox']
        confidence = box['confidence']
        label = box['label']

        detections.append((x1, y1, x2, y2, label, confidence))

        # 繪製邊框與標籤
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)  # 綠色邊框
        cv2.putText(frame, f'{label} {confidence:.2f}', (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 0), 1)

    # 檢查排序是否正確        
    ## 按 x 座標排序
    detections.sort(key=lambda x: x[0])

        
    detect_status = generate_code(detections, names_to_code)
    dete_hist.append(detect_status)
    order_status = code_to_surgery_name(detect_status)
    
    y_offset = 20
    # 顯示排序狀態（手術名稱）
    cv2.putText(frame, f'Sequence Status:', (30, int(height*2//21)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    cv2.putText(frame, f'{order_status}', (30, int(height*2//21)+y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    # 顯示結果
    cv2.imshow("YOLOv11 Real-Time Detection", frame)
    
    k = cv2.waitKey(33)
    if k == ord('q'):
        break
    elif k == ord('a') :
        
        cv2.imwrite(f"new_imgs/{int(time.time())}.png", org_frame)
#         out = cv2.VideoWriter(output_video_name, fourcc, 20.0, (frame.shape[1], frame.shape[0]))
#         recording = True
#         print("開始錄製...")
    if recording:
        out.write(frame)


cap.release()
if recording:
    out.release()
cv2.destroyAllWindows()

/home/mycena/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


按下 'a' 開始錄製，'q' 停止錄製並退出

0: 480x640 7 Needles, 4 mosquitos, 1 thread, 680.4ms
Speed: 2.1ms preprocess, 680.4ms inference, 6.1ms postprocess per image at shape (1, 3, 480, 640)



0: 480x640 7 Needles, 4 mosquitos, 1 thread, 867.7ms
Speed: 7.7ms preprocess, 867.7ms inference, 4.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 Needles, 4 mosquitos, 1 thread, 688.9ms
Speed: 2.2ms preprocess, 688.9ms inference, 4.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 7 Needles, 1 Teeth_Forceps, 4 mosquitos, 1 thread, 683.3ms
Speed: 14.4ms preprocess, 683.3ms inference, 9.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 6 Needles, 4 mosquitos, 1 thread, 934.9ms
Speed: 1.8ms preprocess, 934.9ms inference, 4.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 6 Needles, 4 mosquitos, 671.6ms
Speed: 3.5ms preprocess, 671.6ms inference, 7.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 7 Needles, 4 mosquitos, 1 thread, 682.5ms
Speed: 1.7ms preprocess, 682.5ms inference, 5.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 6 Needles, 4 mosquitos, 1 thread, 823.6ms
Speed: 2.5ms preprocess, 82


0: 480x640 1 Knife_Handle, 11 Needles, 1 Teeth_Forceps, 3 mosquitos, 1 thread, 745.2ms
Speed: 4.3ms preprocess, 745.2ms inference, 7.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 11 Needles, 1 Teeth_Forceps, 3 mosquitos, 1 thread, 744.4ms
Speed: 1.5ms preprocess, 744.4ms inference, 8.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 12 Needles, 1 Teeth_Forceps, 3 mosquitos, 1 thread, 888.5ms
Speed: 1.6ms preprocess, 888.5ms inference, 16.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 11 Needles, 1 Teeth_Forceps, 3 mosquitos, 1 thread, 1073.5ms
Speed: 17.7ms preprocess, 1073.5ms inference, 13.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 11 Needles, 1 Teeth_Forceps, 3 mosquitos, 1 thread, 742.9ms
Speed: 2.2ms preprocess, 742.9ms inference, 7.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 9 Needles, 1 Teeth_Forceps, 3 mosquit


0: 480x640 1 Knife_Handle, 9 Needles, 1 Teeth_Forceps, 3 mosquitos, 1 thread, 896.3ms
Speed: 2.5ms preprocess, 896.3ms inference, 7.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 11 Needles, 1 Teeth_Forceps, 3 mosquitos, 1 thread, 692.4ms
Speed: 1.3ms preprocess, 692.4ms inference, 8.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 12 Needles, 1 Teeth_Forceps, 3 mosquitos, 1 thread, 726.5ms
Speed: 1.7ms preprocess, 726.5ms inference, 8.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 10 Needles, 1 Teeth_Forceps, 3 mosquitos, 1 thread, 712.9ms
Speed: 1.4ms preprocess, 712.9ms inference, 8.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 11 Needles, 1 Teeth_Forceps, 3 mosquitos, 1 thread, 728.4ms
Speed: 12.8ms preprocess, 728.4ms inference, 8.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 12 Needles, 1 Teeth_Forceps, 3 mosquitos, 

...

Speed: 1.5ms preprocess, 658.0ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Needle_Holder, 643.5ms
Speed: 2.2ms preprocess, 643.5ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 654.9ms
Speed: 1.5ms preprocess, 654.9ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Metzenbaum, 1 Needle, 3 Teeth_Forcepss, 654.5ms
Speed: 1.6ms preprocess, 654.5ms inference, 3.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 Teeth_Forcepss, 658.8ms
Speed: 6.1ms preprocess, 658.8ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 Teeth_Forcepss, 638.2ms
Speed: 4.3ms preprocess, 638.2ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 Teeth_Forcepss, 658.2ms
Speed: 1.6ms preprocess, 658.2ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 Teeth_Forcepss, 665.2ms
Speed: 1.5ms preprocess, 6


0: 480x640 1 Knife_Handle, 2 Needle_Holders, 3 Teeth_Forcepss, 2 mosquitos, 646.6ms
Speed: 3.2ms preprocess, 646.6ms inference, 4.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 4 Teeth_Forcepss, 1 mosquito, 688.9ms
Speed: 9.2ms preprocess, 688.9ms inference, 3.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 1 Metzenbaum, 3 Teeth_Forcepss, 630.0ms
Speed: 1.7ms preprocess, 630.0ms inference, 3.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 3 Teeth_Forcepss, 632.0ms
Speed: 7.0ms preprocess, 632.0ms inference, 2.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 3 Teeth_Forcepss, 629.7ms
Speed: 4.4ms preprocess, 629.7ms inference, 2.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 2 Needles, 3 Teeth_Forcepss, 1 thread, 663.6ms
Speed: 3.8ms preprocess, 663.6ms inference, 3.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x64


0: 480x640 1 Needle_Holder, 2 Teeth_Forcepss, 1 mosquito, 654.7ms
Speed: 2.2ms preprocess, 654.7ms inference, 3.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Needle_Holder, 2 Teeth_Forcepss, 1 mosquito, 655.2ms
Speed: 2.8ms preprocess, 655.2ms inference, 2.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Needle_Holder, 2 Teeth_Forcepss, 1 mosquito, 653.3ms
Speed: 1.7ms preprocess, 653.3ms inference, 2.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Needle_Holder, 1 Teeth_Forceps, 2 mosquitos, 651.1ms
Speed: 3.6ms preprocess, 651.1ms inference, 2.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 Needles, 1 Needle_Holder, 2 Teeth_Forcepss, 1 mosquito, 640.7ms
Speed: 4.8ms preprocess, 640.7ms inference, 3.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 2 Needles, 1 Needle_Holder, 3 Teeth_Forcepss, 1 mosquito, 630.2ms
Speed: 1.1ms preprocess, 630.2ms inference, 4.4ms postprocess per imag

Speed: 3.4ms preprocess, 669.9ms inference, 3.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Needle_Holder, 1 Smooth_Forceps, 2 Teeth_Forcepss, 1 mosquito, 666.3ms
Speed: 1.7ms preprocess, 666.3ms inference, 3.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Needle_Holder, 1 Smooth_Forceps, 2 Teeth_Forcepss, 1 mosquito, 652.3ms
Speed: 1.5ms preprocess, 652.3ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Needle_Holder, 1 Smooth_Forceps, 2 Teeth_Forcepss, 1 mosquito, 634.4ms
Speed: 1.2ms preprocess, 634.4ms inference, 3.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Needle_Holder, 1 Smooth_Forceps, 2 Teeth_Forcepss, 1 mosquito, 622.9ms
Speed: 1.3ms preprocess, 622.9ms inference, 3.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Needle_Holder, 1 Smooth_Forceps, 2 Teeth_Forcepss, 1 mosquito, 665.3ms
Speed: 1.4ms preprocess, 665.3ms inference, 3.0ms postprocess per image at shape (1, 

Speed: 1.5ms preprocess, 632.7ms inference, 3.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Needle, 1 Teeth_Forceps, 2 mosquitos, 648.2ms
Speed: 6.4ms preprocess, 648.2ms inference, 2.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Knife_Handle, 2 Needles, 1 Needle_Holder, 1 Teeth_Forceps, 1 mosquito, 668.2ms
Speed: 1.7ms preprocess, 668.2ms inference, 3.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Teeth_Forceps, 1 mosquito, 647.2ms
Speed: 1.4ms preprocess, 647.2ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Teeth_Forceps, 736.6ms
Speed: 3.8ms preprocess, 736.6ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Teeth_Forceps, 631.5ms
Speed: 9.3ms preprocess, 631.5ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Teeth_Forceps, 644.9ms
Speed: 11.2ms preprocess, 644.9ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)


In [ ]:
# 用來檢查到底偵測到的東西是什麼排列
# mylist = list(dict.fromkeys(dete_hist))
# mylist